# Week 6 — Feature engineering and school district geography

Extend the Week 5 model comparison on the **same 130,274 training and 12,470 April 2026 test sales** from `csv/cleaned_sales.csv`. The 41-feature baseline already includes `PropertyAge` in years and an MLS-provided, training-encoded `HighSchoolDistrict` field. This notebook adds a bed/bath ratio and an independent spatial school district layer, then refits the Week 5 linear regression, decision tree, and random forest with both feature sets.

The geographic source is the California Department of Education's [California School District Areas 2024–25](https://data.ca.gov/@california-department-of-education/california-school-district-areas-2024-25) ([official GeoJSON download](https://gis.data.ca.gov/api/download/v1/items/b0e3b936426a47ce9d9a2e77e2bb86cc/geojson?layers=0)). It has Elementary, High, and Unified boundaries. An address can lie in both an elementary and high district, so the two service levels are joined separately; a Unified polygon supplies both. Coordinates use longitude/latitude (CRS84). The file is downloaded to the ignored `data/` directory on first run.

Install `requirements-week6.txt`, run this notebook from the repository root, and generate `csv/cleaned_sales.csv` with `02_preprocessing.ipynb` if it is missing. The boundary vintage precedes part of the 2025–26 sales period, so changes to district boundaries after 2024–25 are not represented.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import TargetEncoder
from sklearn.tree import DecisionTreeRegressor

ROOT = Path.cwd()
DATA_PATH = ROOT / 'csv' / 'cleaned_sales.csv'
if not DATA_PATH.is_file():
    raise FileNotFoundError('Run 02_preprocessing.ipynb to create csv/cleaned_sales.csv.')
sys.path.insert(0, str(ROOT / 'py'))
from school_district_features import join_school_districts

## 1. Reuse the Week 5 rows and feature set

`ClosePrice` is the label; `CloseDate` and `DataSplit` are audit fields. The split, rows, and original numeric predictors stay fixed across both experiments. This is the same holdout already inspected in Week 5, so it is a comparison on an existing test period rather than a new blind test.

In [2]:
data = pd.read_csv(DATA_PATH, parse_dates=['CloseDate'])
metadata_columns = ['CloseDate', 'ClosePrice', 'DataSplit']
missing = set(metadata_columns) - set(data.columns)
if missing:
    raise ValueError(f'Missing metadata columns: {sorted(missing)}')
if set(data['DataSplit']) != {'train', 'test'}:
    raise ValueError('DataSplit must contain exactly train and test.')

old_columns = [c for c in data.columns if c not in metadata_columns]
if len(old_columns) != 41 or 'PropertyAge' not in old_columns:
    raise ValueError('Expected the 41-feature Week 5 export, including PropertyAge.')
if not {'BedroomsTotal', 'BathroomsTotalInteger', 'Latitude', 'Longitude',
        'missingindicator_Latitude', 'missingindicator_Longitude'}.issubset(data.columns):
    raise ValueError('Required property and coordinate fields are missing.')
non_numeric = [c for c in old_columns if not pd.api.types.is_numeric_dtype(data[c])]
if non_numeric:
    raise ValueError(f'Baseline features must be numeric: {non_numeric}')

X_old = data.loc[:, old_columns].copy()
y = pd.to_numeric(data['ClosePrice'], errors='raise')
if not np.isfinite(X_old.to_numpy(dtype=float)).all():
    raise ValueError('Baseline features must all be finite.')
if not np.isfinite(y.to_numpy(dtype=float)).all() or not y.gt(0).all():
    raise ValueError('ClosePrice must be finite and positive.')
if data['CloseDate'].isna().any():
    raise ValueError('CloseDate cannot be missing.')

train_mask = data['DataSplit'].eq('train')
test_mask = data['DataSplit'].eq('test')
if data.loc[train_mask, 'CloseDate'].max() >= data.loc[test_mask, 'CloseDate'].min():
    raise ValueError('Training sales must predate test sales.')
y_train, y_test = y.loc[train_mask], y.loc[test_mask]

split_audit = data.groupby('DataSplit', sort=False).agg(
    rows=('ClosePrice', 'size'),
    first_close=('CloseDate', 'min'),
    last_close=('CloseDate', 'max'),
)
display(split_audit)
print(f'Original model features: {len(old_columns)} (PropertyAge already included)')

,rows,first_close,last_close
DataSplit,,,
train,130274,2025-04-01,2026-03-31
test,12470,2026-04-01,2026-04-30


Original model features: 41 (PropertyAge already included)


## 2. Spatially join each property to school districts

The helper checks the official file and queries its polygons with a spatial index. It preserves one row per sale and returns separate elementary and high district `CDCode` values. It does not assign a district when either coordinate was imputed in Week 3, even if the imputed numeric value falls inside a polygon. An unmatched point remains an explicit category. `CDCode` stays as text so leading zeros are preserved.

In [3]:
district_codes, district_audit = join_school_districts(
    data,
    cache_path=ROOT / 'data' / 'ca_school_district_areas_2024_25.geojson',
)
expected_district_columns = ['ElementaryDistrictCode', 'HighDistrictCode']
if len(district_codes) != len(data) or not district_codes.index.equals(data.index):
    raise AssertionError('Spatial join changed the sale rows or their order.')
if list(district_codes.columns) != expected_district_columns:
    raise AssertionError('Unexpected school district columns.')

district_coverage = pd.DataFrame({
    'Level': ['Elementary or unified', 'High or unified'],
    'Matched sales': [district_codes[c].notna().sum() for c in expected_district_columns],
    'Coverage (%)': [100 * district_codes[c].notna().mean()
                     for c in expected_district_columns],
    'Unique CDCode': [district_codes[c].nunique(dropna=True)
                      for c in expected_district_columns],
})
display(district_coverage.round({'Coverage (%)': 2}))
display(pd.Series(district_audit, name='value').to_frame())

,Level,Matched sales,Coverage (%),Unique CDCode
0,Elementary or unified,142480,99.82,680
1,High or unified,142382,99.75,395


,value
boundary_feature_count,937
boundary_type_counts,"{'Elementary': 516, 'High': 76, 'Unified': 345}"
repaired_boundary_count,7
source_sha256,a57442a35f7238ec30f97270e39ad14a94706a8d352254...
source_matches_reference_sha256,True
source_url,https://gis.data.ca.gov/api/download/v1/items/...
source_page,https://data.ca.gov/@california-department-of-...
boundary_year,2024-25
input_rows,142744
eligible_coordinate_rows,142493


## 3. Engineer model-ready features using training rows only

`BedBathRatio = BedroomsTotal / BathroomsTotalInteger` when bathrooms are positive. Zero-bath records receive the **training** median ratio. The two district IDs are categorical: five-fold cross-fitted target encoding creates training values, and the encoder fitted on training rows alone transforms test values. Encoding uses `log1p(ClosePrice)` to keep the geographic values on the scale of the existing categorical features. Test prices never enter feature construction.

In [4]:
bedrooms = pd.to_numeric(data['BedroomsTotal'], errors='raise')
bathrooms = pd.to_numeric(data['BathroomsTotalInteger'], errors='raise')
raw_ratio = bedrooms.div(bathrooms.where(bathrooms.gt(0))).replace(
    [np.inf, -np.inf], np.nan
)
ratio_fill = float(raw_ratio.loc[train_mask].median())
if not np.isfinite(ratio_fill):
    raise ValueError('Cannot compute the training bed/bath median.')
bed_bath_ratio = raw_ratio.fillna(ratio_fill)

district_categories = district_codes.loc[:, expected_district_columns].fillna(
    '__unmatched__'
).astype(str)
district_encoder = TargetEncoder(
    target_type='continuous', smooth=20.0, cv=5,
    shuffle=True, random_state=42,
)
train_district_values = district_encoder.fit_transform(
    district_categories.loc[train_mask], np.log1p(y_train)
)
test_district_values = district_encoder.transform(
    district_categories.loc[test_mask]
)

X_new = X_old.copy()
X_new['BedBathRatio'] = bed_bath_ratio.to_numpy()
encoded_columns = ['ElementaryDistrictLogPrice', 'HighDistrictLogPrice']
for position, column in enumerate(encoded_columns):
    X_new[column] = np.nan
    X_new.loc[train_mask, column] = train_district_values[:, position]
    X_new.loc[test_mask, column] = test_district_values[:, position]
if not np.isfinite(X_new.to_numpy(dtype=float)).all():
    raise ValueError('Engineered model features must all be finite.')
if list(X_new.columns[:len(old_columns)]) != old_columns:
    raise AssertionError('Original Week 5 model features changed.')

feature_sets = pd.DataFrame([
    {
        'Feature set': 'Week 5 original',
        'Existing features': len(old_columns),
        'PropertyAge (years)': 'included',
        'BedBathRatio': '—',
        'Elementary district from polygons': '—',
        'High district from polygons': '—',
        'Total numeric model features': X_old.shape[1],
    },
    {
        'Feature set': 'Week 6 engineered',
        'Existing features': len(old_columns),
        'PropertyAge (years)': 'included',
        'BedBathRatio': 'added',
        'Elementary district from polygons': 'target encoded CDCode',
        'High district from polygons': 'target encoded CDCode',
        'Total numeric model features': X_new.shape[1],
    },
])
display(feature_sets)
print(f'Nonpositive bathroom rows given training median ratio: {int(bathrooms.le(0).sum()):,}')
print(f'Training median bed/bath ratio: {ratio_fill:.3f}')

,Feature set,Existing features,PropertyAge (years),BedBathRatio,Elementary district from polygons,High district from polygons,Total numeric model features
0,Week 5 original,41,included,—,—,—,41
1,Week 6 engineered,41,included,added,target encoded CDCode,target encoded CDCode,44


Nonpositive bathroom rows given training median ratio: 32
Training median bed/bath ratio: 1.500


## 4. Refit and compare the Week 5 models

Model settings, row membership, and test labels match Week 5. The table shows each model with the original 41 features and the engineered set. Deltas compare each model to its **own** original-feature result; a negative MAE delta means an improvement.

In [5]:
def make_models():
    return {
        'Linear regression': LinearRegression(),
        'Decision tree': DecisionTreeRegressor(
            max_depth=16, min_samples_leaf=20, random_state=42
        ),
        'Random forest': RandomForestRegressor(
            n_estimators=100, max_depth=20, min_samples_leaf=5,
            max_features=0.8, n_jobs=4, random_state=42,
        ),
    }

result_rows = []
for feature_set_name, matrix in [
    ('Week 5 original', X_old), ('Week 6 engineered', X_new)
]:
    X_train, X_test = matrix.loc[train_mask], matrix.loc[test_mask]
    for model_name, model in make_models().items():
        started = perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = perf_counter() - started
        predicted_train = model.predict(X_train)
        predicted_test = model.predict(X_test)
        result_rows.append({
            'Model': model_name,
            'Feature set': feature_set_name,
            'Features': matrix.shape[1],
            'Train R²': r2_score(y_train, predicted_train),
            'Test R²': r2_score(y_test, predicted_test),
            'Test MAE ($)': mean_absolute_error(y_test, predicted_test),
            'Test RMSE ($)': np.sqrt(mean_squared_error(y_test, predicted_test)),
            'Negative test predictions': int(np.sum(predicted_test < 0)),
            'Fit time (s)': fit_seconds,
        })
        print(f'{feature_set_name}: {model_name} fitted in {fit_seconds:.1f} s')

comparison = pd.DataFrame(result_rows).set_index(['Model', 'Feature set'])
for model_name in comparison.index.get_level_values('Model').unique():
    old = comparison.loc[(model_name, 'Week 5 original')]
    for feature_set_name in ('Week 5 original', 'Week 6 engineered'):
        key = (model_name, feature_set_name)
        comparison.loc[key, 'Δ Test R² vs old'] = (
            comparison.loc[key, 'Test R²'] - old['Test R²']
        )
        comparison.loc[key, 'Δ Test MAE ($) vs old'] = (
            comparison.loc[key, 'Test MAE ($)'] - old['Test MAE ($)']
        )
comparison = comparison.loc[:, [
    'Features', 'Train R²', 'Test R²', 'Δ Test R² vs old',
    'Test MAE ($)', 'Δ Test MAE ($) vs old', 'Test RMSE ($)',
    'Negative test predictions', 'Fit time (s)',
]]
display(comparison.round({
    'Train R²': 4, 'Test R²': 4, 'Δ Test R² vs old': 4,
    'Test MAE ($)': 0, 'Δ Test MAE ($) vs old': 0,
    'Test RMSE ($)': 0, 'Fit time (s)': 1,
}))

Week 5 original: Linear regression fitted in 0.2 s


Week 5 original: Decision tree fitted in 2.2 s


Week 5 original: Random forest fitted in 51.7 s


Week 6 engineered: Linear regression fitted in 0.2 s


Week 6 engineered: Decision tree fitted in 2.6 s


Week 6 engineered: Random forest fitted in 57.4 s


,,Features,Train R²,Test R²,Δ Test R² vs old,Test MAE ($),Δ Test MAE ($) vs old,Test RMSE ($),Negative test predictions,Fit time (s)
Model,Feature set,,,,,,,,,
Linear regression,Week 5 original,41,0.6584,0.6632,0.0000,389746.0,0.0,786908.0,623,0.2
Decision tree,Week 5 original,41,0.8521,0.7867,0.0000,233533.0,0.0,626181.0,0,2.2
Random forest,Week 5 original,41,0.9209,0.8366,0.0000,194531.0,0.0,548057.0,0,51.7
Linear regression,Week 6 engineered,44,0.6697,0.6775,0.0143,379820.0,-9926.0,769963.0,555,0.2
Decision tree,Week 6 engineered,44,0.8569,0.8001,0.0134,230938.0,-2595.0,606254.0,0,2.6
Random forest,Week 6 engineered,44,0.9232,0.8405,0.0039,193593.0,-938.0,541512.0,0,57.4


## 5. Read the differences

The following summary is computed from the fitted results, so it stays current when the notebook is rerun. Differences are descriptive for closed MLS sales in the reused April 2026 holdout. The 2024–25 boundary snapshot may misclassify properties whose district changed later, and this notebook does not assess homes outside the MLS history.

In [6]:
for model_name in make_models():
    old = comparison.loc[(model_name, 'Week 5 original')]
    new = comparison.loc[(model_name, 'Week 6 engineered')]
    print(
        f"{model_name}: test R² {old['Test R²']:.4f} → {new['Test R²']:.4f} "
        f"({new['Test R²'] - old['Test R²']:+.4f}); "
        f"MAE ${old['Test MAE ($)']:,.0f} → ${new['Test MAE ($)']:,.0f} "
        f"({new['Test MAE ($)'] - old['Test MAE ($)']:+,.0f})."
    )
best_row = comparison.xs('Week 6 engineered', level='Feature set')['Test MAE ($)'].idxmin()
print(f'Lowest engineered-set test MAE: {best_row}.')

Linear regression: test R² 0.6632 → 0.6775 (+0.0143); MAE $389,746 → $379,820 (-9,926).
Decision tree: test R² 0.7867 → 0.8001 (+0.0134); MAE $233,533 → $230,938 (-2,595).
Random forest: test R² 0.8366 → 0.8405 (+0.0039); MAE $194,531 → $193,593 (-938).
Lowest engineered-set test MAE: Random forest.
